# Add hand-transcribed test data from Ciara

Ciara Ryan <ciara.ryan@met.ie> has already transcribed and QCd some of the Irish daily rainfall sheets during her PhD. These have the same format as the UK ones I'm looking at.

She's sent me her results for 64 images. These are known-good transcriptions, not really enough to be a training dataset, but ideal for validation.


## 1. Get the data into the project format

The originals are in `test_data/from_Ciara/originals`.
They need reformatting to match the `images/transcriptions` layout used here.
This requires PDF and Excel tools, so a separate conda environment is used.


In [ ]:
import subprocess

REPO_ROOT = "../.."  # path from this notebook to the repo root

# Create the special conversion environment (safe to re-run; conda skips if it already exists)
subprocess.run(
    [
        "conda", "env", "create",
        "--file", "test_data/from_Ciara/ciara_convert.yml",
    ],
    cwd=REPO_ROOT,
    check=False,  # don't fail if env already exists
)


In [ ]:

# Run the conversion script inside that environment
subprocess.run(
    [
        "conda", "run", "-n", "ciara-convert",
        "python", "scripts/convert_ciara_test_data.py",
    ],
    cwd=REPO_ROOT,
    check=True,
)

print("Conversion complete.")

## 2. Upload the new test data to Azure

Authenticate with Azure, then push the converted data to the shared datastore.


In [ ]:
import subprocess

# Authenticate (opens a browser window on first use)
subprocess.run(["az", "login"], check=True)

# Upload images and transcriptions to the Azure datastore
subprocess.run(
    [
        "bash", "scripts/aml_upload.sh",
        "--src", "test_data/from_Ciara",
        "--dst", "test_data/from_Ciara",
    ],
    cwd=REPO_ROOT,
    check=True,
)

print("Upload complete.")

## 3. Run a test extraction on the new images

Submit a small extraction job to verify the uploaded images and the current model work end-to-end.


In [ ]:
import subprocess

subprocess.run(
    [
        "bash", "scripts/aml_submit.sh",
        "--images-path", "test_data/from_Ciara/images",
        "--model", "granite4",
        "--limit", "20",
        "--batch-size", "8",
        "extract",
    ],
    cwd=REPO_ROOT,
    check=True,
)

print("Extraction job submitted.")